In [9]:
%cd ..
%reset -f

/home/remi


In [10]:
import pandas as pd
from src.features.blood_cell_metadata import BloodCellMetadata
from src.cfg import BARCELONA_DIR, MUNICH_DIR


df_barcelona = BloodCellMetadata(BARCELONA_DIR,"Barcelona").get_df()
df_Munich = BloodCellMetadata(MUNICH_DIR,"Munich").get_df()
df_all = pd.concat([df_barcelona,df_Munich],ignore_index=True)


df_sample = (df_all.groupby("source", group_keys=False).sample(n=500, random_state=132))


unique = df_sample['source'].unique()
source_mapping = {unique[0]: 0, unique[1]: 1}
df_sample['source'] = df_sample["source"].map(source_mapping).to_numpy()


In [ ]:
from sklearn.model_selection import train_test_split
train_df, test_df = train_test_split(df_sample, test_size=0.15, stratify=df_sample["source"], random_state=132)
train_df, val_df = train_test_split(train_df, test_size=0.15, stratify=train_df["source"], random_state=132)

In [ ]:
import cv2
import numpy as np
from src.features.transform import *
from src.features.common import load_image

transform_barcelone = Compose([
    Crop(min_area= 500.0 , max_center_distance=0.5, margin=0.2, min_size=224),
    Resize((224, 224)),
    Reinhard(load_image("data/PKG-AML-Cytomorphology_LMU/BAS/BAS_0008.tiff")),
    ToFloat32()
], label=0)

transform_munich = Compose([
    Crop(min_area= 500.0 , max_center_distance=0.5, margin=0.2, min_size=224),
    Resize((224, 224)),
    ToFloat32()
], label=1)

transforms = [transform_barcelone,transform_munich]

In [ ]:
from src.features.blood_cell_dataset import BloodCellDataset

train_dataset = BloodCellDataset(metadata=train_df, label="source", transforms=transforms, batch_size=4, shuffle=False,)
val_dataset = BloodCellDataset(metadata=val_df, label="source", transforms=transforms, batch_size=4, shuffle=False)
test_dataset = BloodCellDataset(metadata=test_df, label="source", transforms=transforms, batch_size=4, shuffle=False)

In [14]:
from src.features.CNN_binaire import CNNBinarie

cnn = CNNBinarie(train_dataset,val_dataset, test_dataset, epochs=10)

Training:  90%|█████████ | 9/10 [00:40<00:04,  4.55s/epoch, acc=0.955, loss=0.143, val_acc=0.953]


In [15]:

cnn.resultat()

Accuracy : 0.987
AUC : 0.998


'              precision    recall  f1-score   support\n\n         PBC       1.00      0.97      0.99        75\n      Munich       0.97      1.00      0.99        75\n\n    accuracy                           0.99       150\n   macro avg       0.99      0.99      0.99       150\nweighted avg       0.99      0.99      0.99       150\n'

array([[73,  2],
       [ 0, 75]])